In [1]:
#!pip3 install --upgrade boto3   

> **Actualizado en noviembre 2026.** Amazon Titan Text y Anthropic Claude 3.x llegaron a su fin de vida en Amazon Bedrock (ver [Model lifecycle](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html)). Este notebook usa ahora [Claude Haiku 5.5](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-anthropic-claude-haiku-5-5.html) a través del perfil de inferencia global. Claude 4.7+ ya no acepta `temperature`, `top_p` ni `top_k` ([docs de Anthropic](https://platform.claude.com/docs/en/models/haiku-5-5/overview)), por eso `inferenceConfig` solo lleva `maxTokens`. Claude 5.5 tiene [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) activado por defecto: la respuesta puede empezar con un bloque `reasoningContent` antes del bloque `text`, por eso el código toma los bloques de texto y no `content[0]`. Como estos ejemplos piden respuestas cortas con pocos tokens, se desactiva el [adaptive thinking](https://platform.claude.com/docs/en/build-with-claude/thinking) con `additionalModelRequestFields={"thinking": {"type": "disabled"}}`; si no, el modelo puede gastar todo `maxTokens` pensando y no devolver texto.

# Mejores prácticas para diseñar prompts efectivos

In [2]:
import boto3
import json


In [3]:
bedrock_runtime = boto3.client(
    service_name='bedrock-runtime', 
    region_name='us-east-1'
)

In [4]:
def api_bedrock(prompt):
    response = bedrock_runtime.converse(
    modelId="global.anthropic.claude-haiku-5-5",
    messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "text": prompt
                        }
                    ]
                }
            ],
    # Claude 4.7+ no acepta temperature ni topP: https://platform.claude.com/docs/en/models/haiku-5-5/overview
    inferenceConfig = {
          "maxTokens":250
          },
    # https://platform.claude.com/docs/en/build-with-claude/thinking
    additionalModelRequestFields = {"thinking": {"type": "disabled"}}
          )

    try:
        result = "".join(block["text"] for block in response['output']['message']['content'] if "text" in block) \
        + '\n--- Latency: ' + str(response['metrics']['latencyMs']) \
        + 'ms - Input tokens:' + str(response['usage']['inputTokens']) \
        + ' - Output tokens:' + str(response['usage']['outputTokens']) + ' ---\n'
        return result
    except Exception as e:
        print(e)
        result = "Output parsing error"
    print(result)
    return result

## Ser claro y conciso


In [5]:
prompt1="Escribe un haiku sobre robots"
result = api_bedrock(prompt1)
print(result)

Metal que piensa,
en su corazón de cables
late un sueño frío.
--- Latency: 1411ms - Input tokens:19 - Output tokens:28 ---



In [6]:
prompt2="""¿Escribe un haiku sobre robots. 
            Omite el preámbulo y pasa directamente al poema."""
result = api_bedrock(prompt2)
print(result)

Metal que despierta,
ojos de luz en la noche,
sueña el silicio.
--- Latency: 1128ms - Input tokens:45 - Output tokens:31 ---



## Incluya contexto si es necesario


In [7]:
text = """Amazon Q Developer is a generative artificial intelligence (AI) 
powered conversational assistant that can help you understand, build, extend, 
and operate AWS applications. You can ask questions about AWS architecture, 
your AWS resources, best practices, documentation, support, and more. 
Amazon Q is constantly updating its capabilities so your questions get 
the most contextually relevant and actionable answers.

When used in an integrated development environment (IDE), Amazon Q provides 
software development assistance. Amazon Q can chat about code, provide inline 
code completions, generate net new code, scan your code for security vulnerabilities, 
and make code upgrades and improvements, such as language updates, debugging, and optimizations.

Amazon Q is powered by Amazon Bedrock, a fully managed service that makes 
foundation models (FMs) available through an API. The model that powers Amazon Q 
has been augmented with high quality AWS content to get you more complete, actionable, 
and referenced answers to accelerate your building on AWS."""

In [8]:
prompt1="Resumeme el articulo: "+text
result = api_bedrock(prompt1)
print(result)

Amazon Q Developer es un asistente conversacional impulsado por IA generativa que ayuda a entender, construir, extender y operar aplicaciones de AWS. Permite hacer preguntas sobre arquitectura de AWS, recursos, mejores prácticas, documentación y soporte, y sus capacidades se actualizan constantemente para ofrecer respuestas relevantes y accionables.

Cuando se usa en un entorno de desarrollo integrado (IDE), ofrece asistencia para programar: conversar sobre código, sugerir autocompletado en línea, generar código nuevo, analizar el código en busca de vulnerabilidades de seguridad y realizar mejoras y actualizaciones, como cambios de versión de len
--- Latency: 1711ms - Input tokens:357 - Output tokens:250 ---



In [9]:
prompt2="Proporcione un resumen de este artículo para utilizarlo en una publicación de blog: "+text
result = api_bedrock(prompt2)
print(result)

**Amazon Q Developer: tu asistente de IA para AWS**

Amazon Q Developer es un asistente conversacional impulsado por inteligencia artificial generativa que te ayuda a entender, construir, ampliar y operar aplicaciones en AWS. Puedes hacerle preguntas sobre arquitectura de AWS, tus recursos, buenas prácticas, documentación, soporte y mucho más. Como se actualiza constantemente, sus respuestas son cada vez más relevantes y prácticas según el contexto.

Dentro de un entorno de desarrollo integrado (IDE), Amazon Q ofrece asistencia para programar. Puede conversar sobre código, sugerir autocompletados en línea, generar código
--- Latency: 1757ms - Input tokens:377 - Output tokens:250 ---



## USA XML tags

In [10]:
text = """Amazon Q Developer is a generative artificial intelligence (AI) 
powered conversational assistant that can help you understand, build, extend, 
and operate AWS applications. You can ask questions about AWS architecture, 
your AWS resources, best practices, documentation, support, and more. 
Amazon Q is constantly updating its capabilities so your questions get 
the most contextually relevant and actionable answers.

When used in an integrated development environment (IDE), Amazon Q provides 
software development assistance. Amazon Q can chat about code, provide inline 
code completions, generate net new code, scan your code for security vulnerabilities, 
and make code upgrades and improvements, such as language updates, debugging, and optimizations.

Amazon Q is powered by Amazon Bedrock, a fully managed service that makes 
foundation models (FMs) available through an API. The model that powers Amazon Q 
has been augmented with high quality AWS content to get you more complete, actionable, 
and referenced answers to accelerate your building on AWS."""
RED_SOCIAL = "linkedin"

In [11]:
prompt=f""" Eres un experto en publicidad en redes sociales, te entregan el texto y la red social y creas la publicidad. 
<texto>{text}</texto>
<red_social>{RED_SOCIAL}</red_social>
"""
result = api_bedrock(prompt)
print(result)

**Publicación para LinkedIn: Amazon Q Developer**

🚀 ¿Cansado de buscar documentación de AWS a medianoche? Amazon Q Developer es tu nuevo copiloto.

Es un asistente conversacional impulsado por IA generativa que te ayuda a **entender, construir, extender y operar** tus aplicaciones en AWS. Pregúntale sobre:

✅ Arquitectura de AWS
✅ Tus recursos y mejores prácticas
✅ Documentación y soporte

💻 **Dentro de tu IDE**, va más allá:
→ Conversa sobre tu código
→ Completa código en línea y genera código nuevo
→ Detecta vulnerabilidades de
--- Latency: 2064ms - Input tokens:411 - Output tokens:250 ---



## Zero-Shot

In [12]:
prompt=f""" Clasifique el texto en neutro, negativo o positivo.
Texto: Creo que Mazatlán esta increíble.
Sentimiento:
"""
result = api_bedrock(prompt)
print(result)

**Positivo**

El texto expresa una opinión favorable sobre Mazatlán ("increíble").
--- Latency: 1398ms - Input tokens:57 - Output tokens:37 ---



## Few-Shot

In [13]:
prompt=f""" Clasificar reseñas de restaurantes como positivas o negativas.

Reseña: La comida estaba deliciosa y el servicio fue excelente.
Clasificación: Positiva

Reseña: El lugar estaba sucio y la comida fría.
Clasificación: Negativa

Reseña: Precios razonables y ambiente agradable.
Clasificación:
"""
result = api_bedrock(prompt)
print(result)

Clasificación: Positiva
--- Latency: 4679ms - Input tokens:132 - Output tokens:12 ---



## Chain-of-thought prompting
## Zero-shot

In [14]:
prompt=f"""Problema:
En una fiesta de la Pycon, hay 3 pizzas grandes. Cada pizza está cortada en 8 rebanadas. Si hay 12 invitados y cada uno come 2 rebanadas, ¿cuántas rebanadas de pizza sobrarán?

Instrucción:
Resuelve este problema pensando paso a paso. Muestra tu razonamiento en cada etapa.

"""
result = api_bedrock(prompt)
print(result)

**Paso 1: Calcular el total de rebanadas disponibles**

Cada pizza tiene 8 rebanadas y hay 3 pizzas:

$$3 \times 8 = 24 \text{ rebanadas}$$

**Paso 2: Calcular cuántas rebanadas comen los invitados**

Hay 12 invitados y cada uno come 2 rebanadas:

$$12 \times 2 = 24 \text{ rebanadas}$$

**Paso 3: Calcular las rebanadas sobrantes**

Restamos las rebanadas consumidas del total disponible:

$$24 - 24 = 0 \text{ rebanadas}$$

**Respuesta:** No sobrará ninguna rebanada
--- Latency: 2074ms - Input tokens:147 - Output tokens:250 ---



## Chain-of-thought prompting
## Few-shot

In [15]:
prompt=f"""Tarea: Calcular cuántos tacos puede comer un programador en una noche de coding.
Datos: 3 horas de coding, 1 bug resuelto
Pensamiento: 
1. Base: 3 tacos/hora x 3 horas = 9 tacos 
2. Bonus por bug: +2 tacos
Total: 11 tacos
Datos: 2 horas de coding, 1 feature completada
Pensamiento: 
1. Base: 3 tacos/hora x 2 horas = 6 tacos
2. Bonus por feature: +3 tacos
Total: 9 tacos

Datos: 4 horas de coding, 2 bugs resueltos, 1 crash del sistema:

"""
result = api_bedrock(prompt)
print(result)

Siguiendo la misma lógica del ejemplo:

**Datos:** 4 horas de coding, 2 bugs resueltos, 1 crash del sistema

**Pensamiento:**
1. Base: 3 tacos/hora x 4 horas = 12 tacos
2. Bonus por bugs: +2 tacos x 2 bugs = +4 tacos
3. Penalización por crash del sistema: -2 tacos (supuesto, ya que el ejemplo no define un valor; puedes ajustarlo)

**Total: 14 tacos**

Nota: como el patrón de los ejemplos no especifica cuánto restar por un crash, asumí -2 tacos. Si prefieres otra penalización, dime el valor y rec
--- Latency: 2194ms - Input tokens:245 - Output tokens:250 ---



## Utilice directivas para el tipo de respuesta deseado


In [16]:
prompt1="¿Cuál es la capital de Nueva York? Proporcione la respuesta en una oración completa."
result = api_bedrock(prompt1)
print(result)

La capital del estado de Nueva York es Albany.
--- Latency: 2181ms - Input tokens:40 - Output tokens:19 ---



In [17]:
prompt2="¿Cuál es la capital de Nueva York?"
result = api_bedrock(prompt2)
print(result)

La capital del estado de Nueva York es **Albany**. No debe confundirse con la ciudad de Nueva York, que es la más poblada del estado y del país, pero no es su capital.
--- Latency: 1186ms - Input tokens:23 - Output tokens:63 ---



## Considere el resultado en el mensaje

In [18]:
prompt1="Calcula el área de un círculo con un radio de 3 pulgadas"
api_bedrock(prompt1)

'El área de un círculo se calcula con la fórmula:\n\n**A = π × r²**\n\nCon un radio de r = 3 pulgadas:\n\n**A = π × (3 pulgadas)²**\n**A = π × 9 pulgadas²**\n**A = 9π pulgadas²**\n\nAproximadamente, usando π ≈ 3.14159:\n\n**A ≈ 28.27 pulgadas²**\n--- Latency: 2024ms - Input tokens:30 - Output tokens:135 ---\n'

In [19]:
prompt2="Calcula el área de un círculo con un radio de 3 pulgadas (7,5 cm). Redondea tu respuesta al número entero más cercano."
api_bedrock(prompt2)

'Área = π × r² = π × (3 pulgadas)² = 9π ≈ 28,27 pulgadas cuadradas.\n\nRedondeado al entero más cercano: **28 pulgadas cuadradas** (≈ 181 cm², ya que 1 pulgada² = 6,4516 cm²).\n--- Latency: 1870ms - Input tokens:57 - Output tokens:106 ---\n'

## Iniciar mensajes con una pregunta

In [20]:
evento = """ En septiembre de 1666, un pequeño incendio comenzó en la panadería del panadero Thomas Farriner en Pudding Lane, en Londres. Sin embargo, debido a las condiciones secas y al viento fuerte, el fuego se extendió rápidamente a través de las casas de madera apiñadas en la ciudad. Dentro de pocas horas, el fuego se había extendido desde Pudding Lane hasta la catedral de San Pablo. Al día siguiente, el fuego había arrasado la mayor parte de la ciudad dentro de las murallas. Durante casi una semana, el fuego ardió sin control. Solo se detuvo cuando el viento cambió de dirección y los esfuerzos para demoler edificios crearon calles de fuego para detener su propagación.
Cuando finalmente terminó, el Gran Incendio de Londres había destruido 13,200 casas, 87 iglesias parroquiales, St. Paul's Cathedral y muchos edificios gubernamentales. Se estima que la pérdida de vidas humanas fue de solo unas pocas docenas, pero decenas de miles de londinenses se quedaron sin hogar. Fue uno de los peores desastres en la historia de Inglaterra. El incendio tuvo un impacto duradero en Londres y en Inglaterra. Sirvió para impulsar la reconstrucción de Londres con amplias calles y edificios de piedra en lugar de madera, reduciendo así el riesgo de futuros incendios a gran escala. También llevó a la creación del London Building Acts, uno de los primeros códigos de construcción del mundo."""

In [21]:
prompt1= "¿Por qué ocurrió el evento en los tags <evento></evento>? Explícalo en tres frases. <evento>"+evento+"</evento>"

api_bedrock(prompt1)

'El Gran Incendio de Londres ocurrió porque un pequeño fuego que empezó en la panadería de Thomas Farriner en Pudding Lane se propagó rápidamente, debido a las condiciones secas, al viento fuerte y a que la ciudad estaba llena de casas de madera apiñadas. El fuego solo se detuvo cuando el viento cambió de dirección y se demolieron edificios para crear cortafuegos, tras casi una semana de incendio incontrolado. El desastre causó enormes daños, y su legado fue la reconstrucción de Londres con calles más amplias y edificios de piedra, así como la creación de las London Building Acts, uno de los primeros códigos de construcción del mundo.\n--- Latency: 4458ms - Input tokens:602 - Output tokens:246 ---\n'

In [22]:
prompt2= "Resume  el evento en los tags <evento></evento>. <evento>"+evento+"</evento>"

api_bedrock(prompt2)

'El Gran Incendio de Londres comenzó en septiembre de 1666, cuando un pequeño fuego se originó en la panadería de Thomas Farriner, en Pudding Lane. Debido a las condiciones secas y al viento fuerte, las casas de madera apiñadas en la ciudad permitieron que el fuego se propagara con rapidez: en pocas horas alcanzó la catedral de San Pablo, y al día siguiente había consumido gran parte de la ciudad dentro de las murallas. Durante casi una semana el incendio ardió sin control, y solo se detuvo cuando el viento cambió de dirección y se demolieron edificios para crear cortafuegos.\n\nAl final, el incendio destruyó unas 13,200\n--- Latency: 2181ms - Input tokens:584 - Output tokens:250 ---\n'

## Proporcione una respuesta de ejemplo (one shot)

In [23]:
review_negativo =  """Compré el nuevo teléfono X pero ha sido una decepción total. La batería dura apenas unas horas y la cámara toma fotos borrosas. Además se cuelga constantemente. No puedo recomendar este producto."""

In [24]:
prompt1= "Determine el sentimiento de esta publicación en las redes sociales:"+review_negativo
api_bedrock(prompt1)

'**Sentimiento: negativo**\n\nLa publicación expresa una insatisfacción clara y fuerte con el teléfono X. Sus puntos principales son:\n\n- **Batería:** dura solo unas horas, lo que indica un problema grave de autonomía.\n- **Cámara:** toma fotos borrosas, una queja sobre la calidad de imagen.\n- **Rendimiento:** el teléfono se cuelga constantemente, lo que señala problemas de estabilidad.\n- **Recomendación:** el autor explícitamente no recomienda el producto.\n\nEl tono es de decepción ("ha sido una decepción total") y de desaprobación, sin matices ni aspectos positivos mencionados.\n--- Latency: 1938ms - Input tokens:105 - Output tokens:236 ---\n'

In [25]:
prompt2 = f"""Determine el sentimiento de la siguiente publicación en las redes sociales utilizando estos ejemplos:
post: “Muy buen producto, me encanta.” sentimiento: Positivo
post: “Odio cuando se agota la batería de mi teléfono.” sentimiento:  Negativo
post: "{review_negativo}" sentimiento:  """
api_bedrock(prompt2)

'**Sentimiento: Negativo**\n\nLa publicación expresa una clara insatisfacción con el producto: califica la compra como "una decepción total", menciona problemas concretos (batería, cámara, bloqueos) y termina con una recomendación negativa.\n--- Latency: 1720ms - Input tokens:193 - Output tokens:86 ---\n'